# S2 / S1 Temporal Gap Analysis

## Event analysis
Closest S1 and S2 images before/after the event date (from GEE, ±120 day window).
Three cloud filtering scenarios compared.

In [2]:
import os
import plotly.express as px
import pandas as pd
import numpy as np

DATA_DIR = os.path.expanduser('~/Desktop/Master/thesis/tropical_forest_disturbance/data_csv')
gap_cols = ['s2BefGap', 's2AftGap', 's1BefGap', 's1AftGap']

scenarios = {
    'v1: QA60+SCL': pd.read_csv(os.path.join(DATA_DIR, 'closest_s1_s2_gap_analysis.csv')),
    'v2: CS+ dilated': pd.read_csv(os.path.join(DATA_DIR, 'closest_s1_s2_gap_csplus_dilated.csv')),
    'v3: CS+ no dilation': pd.read_csv(os.path.join(DATA_DIR, 'closest_s1_s2_gap_csplus_nodilation.csv')),
}

# Replace NaN with ">120" (no image found within ±120 days)
for name, df in scenarios.items():
    for col in gap_cols:
        df[col] = df[col].fillna('>120')

# Summary per scenario
for name, df in scenarios.items():
    print(f"\n{'='*60}")
    print(f"  {name} — {len(df)} polygons")
    print(f"{'='*60}")
    for col in gap_cols:
        found = (df[col] != '>120').sum()
        missing = (df[col] == '>120').sum()
        print(f"  {col:12s}  found: {found:3d}   >120: {missing:3d}")


  v1: QA60+SCL — 389 polygons
  s2BefGap      found: 386   >120:   3
  s2AftGap      found: 389   >120:   0
  s1BefGap      found: 389   >120:   0
  s1AftGap      found: 389   >120:   0

  v2: CS+ dilated — 389 polygons
  s2BefGap      found: 361   >120:  28
  s2AftGap      found: 387   >120:   2
  s1BefGap      found: 389   >120:   0
  s1AftGap      found: 389   >120:   0

  v3: CS+ no dilation — 389 polygons
  s2BefGap      found: 372   >120:  17
  s2AftGap      found: 388   >120:   1
  s1BefGap      found: 389   >120:   0
  s1AftGap      found: 389   >120:   0


In [4]:
def histogram_day_gap(df_evt, title_suffix=''):
    df_long = pd.melt(
        df_evt, id_vars=['fid'],
        value_vars=gap_cols,
        var_name='gap_type', value_name='day_gap',
    )
    df_long['day_gap'] = df_long['day_gap'].apply(lambda x: 121 if x == '>120' else float(x))
    df_long['day_gap'] = df_long['day_gap'].astype(int)
    df_long['gap_type'] = df_long['gap_type'].map({
        's2BefGap': 'S2 before event', 's2AftGap': 'S2 after event',
        's1BefGap': 'S1 before event', 's1AftGap': 'S1 after event',
    })
    fig = px.histogram(
        df_long, x='day_gap', color='gap_type',
        nbins=int(df_long['day_gap'].max()) + 1,
        title=f'Closest S1/S2 image gap to event date{title_suffix} (121 = >120 days)',
        labels={'day_gap': 'Days from event date', 'gap_type': 'Type'},
        barmode='overlay', opacity=0.6,
    )
    fig.update_layout(xaxis_title='Days from event date', yaxis_title='Number of polygons')
    fig.show()

# Plot histogram for each scenario
for name, df in scenarios.items():
    histogram_day_gap(df, f' — {name}')

### Side-by-side S2 gap comparison across all three filters

In [5]:
def make_long(df_src, filter_name):
    df_l = pd.melt(
        df_src, id_vars=['fid'],
        value_vars=['s2BefGap', 's2AftGap'],
        var_name='gap_type', value_name='day_gap',
    )
    df_l['day_gap'] = df_l['day_gap'].apply(lambda x: 121 if x == '>120' else float(x))
    df_l['day_gap'] = df_l['day_gap'].astype(int)
    df_l['gap_type'] = df_l['gap_type'].map({
        's2BefGap': 'S2 before event', 's2AftGap': 'S2 after event',
    })
    df_l['filter'] = filter_name
    return df_l

df_all = pd.concat([make_long(df, name) for name, df in scenarios.items()])

fig = px.histogram(
    df_all, x='day_gap', color='filter', facet_col='gap_type',
    nbins=60,
    title='S2 gap comparison across filters (121 = >120 days)',
    labels={'day_gap': 'Days from event date', 'filter': 'Cloud filter'},
    barmode='overlay', opacity=0.5,
)
fig.update_layout(yaxis_title='Number of polygons')
fig.show()

### Comparative statistics

In [6]:
labels_map = {
    's2BefGap': 'S2 before event', 's2AftGap': 'S2 after event',
    's1BefGap': 'S1 before event', 's1AftGap': 'S1 after event',
}

rows = []
for col in gap_cols:
    for name, df_src in scenarios.items():
        numeric_vals = df_src[col][df_src[col] != '>120'].astype(float)
        n_missing = (df_src[col] == '>120').sum()
        rows.append({
            'Gap type': labels_map[col],
            'Filter': name,
            'Found': len(numeric_vals),
            '>120 days': int(n_missing),
            'Mean': round(numeric_vals.mean(), 1) if len(numeric_vals) > 0 else None,
            'Median': round(numeric_vals.median(), 1) if len(numeric_vals) > 0 else None,
            'P90': round(np.percentile(numeric_vals, 90), 1) if len(numeric_vals) > 0 else None,
            'Max': int(numeric_vals.max()) if len(numeric_vals) > 0 else None,
        })

df_compare = pd.DataFrame(rows)
print("Comparative statistics (389 polygons)")
df_compare

Comparative statistics (389 polygons)


,Gap type,Filter,Found,>120 days,Mean,Median,P90,Max
0,S2 before event,v1: QA60+SCL,386,3,10.2,6.0,21.0,109
1,S2 before event,v2: CS+ dilated,361,28,11.8,7.0,22.0,109
2,S2 before event,v3: CS+ no dilation,372,17,12.1,7.0,23.8,119
3,S2 after event,v1: QA60+SCL,389,0,9.7,6.0,21.0,113
4,S2 after event,v2: CS+ dilated,387,2,11.6,6.0,28.0,113
5,S2 after event,v3: CS+ no dilation,388,1,10.8,6.0,24.0,113
6,S1 before event,v1: QA60+SCL,389,0,8.2,7.0,13.0,24
7,S1 before event,v2: CS+ dilated,389,0,8.2,7.0,13.0,24
8,S1 before event,v3: CS+ no dilation,389,0,8.2,7.0,13.0,24
9,S1 after event,v1: QA60+SCL,389,0,6.9,6.0,16.2,23


### Image quality vs temporal availability: effect of dilation
Polygons where CS+ with dilation rejected an image that CS+ without dilation accepted.
These are images where **only the 10px dilation** caused the rejection.

In [7]:
# Compare v2 (CS+ dilated) vs v3 (CS+ no dilation)
# Shows images where ONLY the dilation caused the rejection
df_v2_raw = pd.read_csv(os.path.join(DATA_DIR, 'closest_s1_s2_gap_csplus_dilated.csv'))
df_v3_raw = pd.read_csv(os.path.join(DATA_DIR, 'closest_s1_s2_gap_csplus_nodilation.csv'))

df_diff = df_v2_raw[['fid', 'evtDate', 's2BefGap', 's2BefId', 's2AftGap', 's2AftId']].merge(
    df_v3_raw[['fid', 's2BefGap', 's2BefId', 's2AftGap', 's2AftId']],
    on='fid', suffixes=('_dilated', '_nodil')
)

total_dilation_only = 0
for direction, gap_col, id_col in [('before', 's2BefGap', 's2BefId'),
                                     ('after', 's2AftGap', 's2AftId')]:
    dil_gap = df_diff[f'{gap_col}_dilated']
    nod_gap = df_diff[f'{gap_col}_nodil']

    # v3 found a closer image than v2 (dilation rejected it)
    mask_closer = nod_gap.notna() & dil_gap.notna() & (nod_gap < dil_gap)
    # v3 found an image but v2 didn't
    mask_found = nod_gap.notna() & dil_gap.isna()
    mask = mask_closer | mask_found

    rejected = df_diff[mask].copy()
    total_dilation_only += len(rejected)

    print(f"\n{'='*70}")
    print(f"  S2 {direction}: {len(rejected)} polygons where dilation rejected a clean image")
    print(f"{'='*70}")

    if len(rejected) > 0:
        rejected['gap_diff'] = dil_gap[mask].fillna(999) - nod_gap[mask]
        rejected = rejected.sort_values('gap_diff', ascending=False)
        print(rejected[['fid', 'evtDate',
                         f'{gap_col}_dilated', f'{gap_col}_nodil',
                         f'{id_col}_nodil', 'gap_diff']].rename(columns={
            f'{gap_col}_dilated': 'gap_dilated',
            f'{gap_col}_nodil': 'gap_no_dil',
            f'{id_col}_nodil': 'image_accepted',
            'gap_diff': 'days_lost',
        }).to_string(index=False))

print(f"\n\nTotal: {total_dilation_only} polygons where dilation was the sole cause of rejection")


  S2 before: 25 polygons where dilation rejected a clean image
 fid    evtDate  gap_dilated  gap_no_dil                         image_accepted  days_lost
 242 2023-04-12          NaN         7.0 20230405T142711_20230405T142712_T20LNR      992.0
 171 2023-05-29          NaN        14.0 20230515T142711_20230515T142713_T20LNR      985.0
 272 2023-03-08          NaN        22.0 20230214T142711_20230214T143425_T20LNR      977.0
  95 2023-03-15          NaN        29.0 20230214T142711_20230214T143425_T20LNR      970.0
 216 2023-03-15          NaN        29.0 20230214T142711_20230214T143425_T20LNR      970.0
 273 2023-03-15          NaN        29.0 20230214T142711_20230214T143425_T20LNR      970.0
 232 2023-05-27          NaN        37.0 20230420T142719_20230420T142714_T20LNR      962.0
 289 2023-05-27          NaN        37.0 20230420T142719_20230420T142714_T20LNR      962.0
 145 2023-03-15          NaN        44.0 20230130T142709_20230130T143158_T20LNR      955.0
 266 2023-03-08          N

### QA60+SCL vs CS+ no dilation
Polygons where CS+ (without dilation) rejected an image that QA60+SCL accepted.
These are images where **Cloud Score+ itself** (not the dilation) decided the image was cloudy/hazy.

In [8]:
# Compare v1 (QA60+SCL) vs v3 (CS+ no dilation)
# Shows images where Cloud Score+ itself rejected (not dilation)
df_v1_raw = pd.read_csv(os.path.join(DATA_DIR, 'closest_s1_s2_gap_analysis.csv'))
df_v3_raw = pd.read_csv(os.path.join(DATA_DIR, 'closest_s1_s2_gap_csplus_nodilation.csv'))

df_diff = df_v1_raw[['fid', 'evtDate', 's2BefGap', 's2BefId', 's2AftGap', 's2AftId']].merge(
    df_v3_raw[['fid', 's2BefGap', 's2BefId', 's2AftGap', 's2AftId']],
    on='fid', suffixes=('_v1', '_v3')
)

total_cs_only = 0
for direction, gap_col, id_col in [('before', 's2BefGap', 's2BefId'),
                                     ('after', 's2AftGap', 's2AftId')]:
    v1_gap = df_diff[f'{gap_col}_v1']
    v3_gap = df_diff[f'{gap_col}_v3']

    # v3 has a larger gap than v1 (CS+ rejected a closer image)
    mask_larger = v1_gap.notna() & v3_gap.notna() & (v3_gap > v1_gap)
    # v1 found an image but v3 didn't
    mask_lost = v1_gap.notna() & v3_gap.isna()
    mask = mask_larger | mask_lost

    rejected = df_diff[mask].copy()
    total_cs_only += len(rejected)

    print(f"\n{'='*70}")
    print(f"  S2 {direction}: {len(rejected)} polygons where CS+ (no dilation) rejected a closer image")
    print(f"{'='*70}")

    if len(rejected) > 0:
        rejected['gap_diff'] = v3_gap[mask].fillna(999) - v1_gap[mask]
        rejected = rejected.sort_values('gap_diff', ascending=False)
        print(rejected[['fid', 'evtDate',
                         f'{gap_col}_v1', f'{id_col}_v1',
                         f'{gap_col}_v3', 'gap_diff']].rename(columns={
            f'{gap_col}_v1': 'gap_QA60',
            f'{id_col}_v1': 'image_rejected_by_CS',
            f'{gap_col}_v3': 'gap_CS_nodil',
            'gap_diff': 'days_lost',
        }).to_string(index=False))

print(f"\n\nTotal: {total_cs_only} polygons where Cloud Score+ itself rejected an image accepted by QA60+SCL")


  S2 before: 58 polygons where CS+ (no dilation) rejected a closer image
 fid    evtDate  gap_QA60                   image_rejected_by_CS  gap_CS_nodil  days_lost
 150 2023-04-13       8.0 20230405T142711_20230405T142712_T20LNR           NaN      991.0
 153 2023-04-29       9.0 20230420T142719_20230420T142714_T20LNR           NaN      990.0
 225 2023-04-29       9.0 20230420T142719_20230420T142714_T20LNR           NaN      990.0
 152 2023-04-19      14.0 20230405T142711_20230405T142712_T20LMR           NaN      985.0
  97 2023-04-19      14.0 20230405T142711_20230405T142712_T20LMR           NaN      985.0
 144 2023-03-15      14.0 20230301T142709_20230301T142737_T20LNR           NaN      985.0
  92 2023-03-15      14.0 20230301T142709_20230301T142737_T20LMR           NaN      985.0
 146 2023-03-15      29.0 20230214T142711_20230214T143425_T20LNR           NaN      970.0
  93 2023-03-15      29.0 20230214T142711_20230214T143425_T20LNR           NaN      970.0
 231 2023-05-27      37.0 

After doing this comparison i can see that 7 days is a good range of time to get the median of the images. So that is gonna be the range to find the event image for S1 and S2. 

For the image of the before the event, based on the literature and the PRODES use a window of the dry season starts on July 31st from the previous year. (?)

For the image after How it is an early detection system, we can use 21, 24, 28? from the different P90 or the median? or 7 as well. 


## Pairing analysis
Day gap between each S2 tile and its paired S1 tile (from CSV, S1 must be on or before S2 date).

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

DATA_DIR = os.path.expanduser('~/Desktop/Master/thesis/tropical_forest_disturbance/data_csv')
CSVS = ['v1_images_s2_s1.csv', 'v2_images_s2_s1.csv', 'v3_images_s2_s1.csv']

# Dates in the CSVs are stored as "YYYY-?-DOY" (year, unused, day-of-year)
def parse_doy_dates(cell):
    if pd.isna(cell) or str(cell).strip() == '':
        return []
    dates = []
    for entry in str(cell).split(','):
        parts = entry.strip().split('-')
        try:
            year, doy = int(parts[0]), int(parts[2])
            dates.append(datetime(year, 1, 1) + timedelta(days=doy - 1))
        except (IndexError, ValueError):
            pass
    return dates

WINDOWS = [('befDatS2', 'befDatS1'),
            ('evtDatS2', 'evtDatS1'),
            ('aftDatS2', 'aftDatS1')]

# For each S2, gap to the closest S1 in time (absolute value = production logic)
gaps = []
for f in CSVS:
    df = pd.read_csv(os.path.join(DATA_DIR, f))
    for _, row in df.iterrows():
        for c_s2, c_s1 in WINDOWS:
            if c_s2 not in df.columns or c_s1 not in df.columns:
                continue
            s2_dates = parse_doy_dates(row[c_s2])
            s1_dates = parse_doy_dates(row[c_s1])
            for s2 in s2_dates:
                if s1_dates:
                    gaps.append(min(abs((s2 - s1).days) for s1 in s1_dates))

gaps = np.array(gaps)

# Summary table (Median / P90 / Max)
table = pd.DataFrame({
    'Sentinel 1 to Sentinel 2 gap': ['Median', '90th percentile', 'Maximum'],
    'Days': [int(np.median(gaps)),
            int(np.percentile(gaps, 90)),
            int(gaps.max())]
})
print(table.to_string(index=False))

# The 95% figure goes in the text, not in the table
pct_within_7 = 100 * (gaps <= 7).mean()
print(f'\nn = {len(gaps)} pairs')
print(f'Pairs within 7 days: {pct_within_7:.0f}%')


Sentinel 1 to Sentinel 2 gap  Days
                      Median     3
             90th percentile     6
                     Maximum    16

n = 16347 pairs
Pairs within 7 days: 95%
